# Fashion Item Classification with CNN
**Project ID:** DL-002  
**Domain:** Computer Vision / Retail  
**Dataset:** Fashion-MNIST (Zalando Research)  

This notebook documents the end-to-end training and evaluation pipeline for the Fashion-MNIST Deep Convolutional Neural Network (CNN). It mirrors the exact architecture, preprocessing, and training configuration implemented in `backend/train.py` and `backend/evaluate.py`.

## 1. Imports and Reproducibility Setup
We fix the random seed to `42` across Python, NumPy, and TensorFlow to ensure deterministic results.

In [ ]:
import os
import random
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_SEED = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"TensorFlow Version: {tf.__version__}")

## 2. Dataset Ingestion & Partitioning
- **Source:** Official Fashion-MNIST dataset (`keras.datasets.fashion_mnist`)
- **Train / Val Split:** 54,000 training images, 6,000 validation images (stratified, random seed 42)
- **Held-Out Test Set:** 10,000 official test images

In [ ]:
CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Load full 60k training and 10k test datasets
(x_train_full, y_train_full), (x_test_raw, y_test) = keras.datasets.fashion_mnist.load_data()

# Stratified split of 60k training set -> 54k train, 6k validation
x_train_raw, x_val_raw, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=6000,
    random_state=RANDOM_SEED,
    stratify=y_train_full
)

# Normalize to [0.0, 1.0] and reshape to (28, 28, 1)
x_train = (x_train_raw.astype("float32") / 255.0).reshape(-1, 28, 28, 1)
x_val = (x_val_raw.astype("float32") / 255.0).reshape(-1, 28, 28, 1)
x_test = (x_test_raw.astype("float32") / 255.0).reshape(-1, 28, 28, 1)

print(f"Training set shape:   {x_train.shape}, labels: {y_train.shape}")
print(f"Validation set shape: {x_val.shape}, labels: {y_val.shape}")
print(f"Test set shape:       {x_test.shape}, labels: {y_test.shape}")

## 3. Sample Visualization
Visualize 10 sample images representing each Fashion-MNIST category.

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    idx = np.where(y_train == i)[0][0]
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[idx].squeeze(), cmap="gray")
    plt.title(CLASS_NAMES[i], fontsize=10)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. CNN Architecture Construction
The CNN architecture follows the exact technical specification:
1. **Input:** `(28, 28, 1)`
2. **Conv2D:** 32 filters, 3×3 kernel, ReLU activation
3. **MaxPooling2D:** 2×2 pool size
4. **Conv2D:** 64 filters, 3×3 kernel, ReLU activation
5. **MaxPooling2D:** 2×2 pool size
6. **Flatten:** Vectorizes 2D feature maps
7. **Dense:** 128 units, ReLU activation
8. **Dropout:** Rate 0.3 (regularization against overfitting)
9. **Dense Output:** 10 units with Softmax activation

In [ ]:
def create_cnn_model():
    model = keras.Sequential([
        keras.Input(shape=(28, 28, 1), name="input_image"),
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu", name="conv2d_1"),
        layers.MaxPooling2D(pool_size=(2, 2), name="maxpool_1"),
        layers.Conv2D(64, kernel_size=(3, 3), activation="relu", name="conv2d_2"),
        layers.MaxPooling2D(pool_size=(2, 2), name="maxpool_2"),
        layers.Flatten(name="flatten"),
        layers.Dense(128, activation="relu", name="dense_features"),
        layers.Dropout(0.3, name="dropout"),
        layers.Dense(10, activation="softmax", name="output_probabilities"),
    ], name="FashionMNIST_CNN")
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"]
    )
    return model

model = create_cnn_model()
model.summary()

## 5. Model Training (15 Epochs, Batch Size 64)

In [ ]:
EPOCHS = 15
BATCH_SIZE = 64

history = model.fit(
    x_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_val, y_val),
    verbose=1
)

## 6. Training History Visualization (Loss & Accuracy Curves)

In [ ]:
epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, [a * 100 for a in history.history["accuracy"]], label="Training Accuracy", marker="o", color="#4F46E5")
plt.plot(epochs_range, [a * 100 for a in history.history["val_accuracy"]], label="Validation Accuracy", marker="s", color="#06B6D4")
plt.title("Accuracy over Epochs", fontsize=12, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history["loss"], label="Training Loss", marker="o", color="#DC2626")
plt.plot(epochs_range, history.history["val_loss"], label="Validation Loss", marker="s", color="#F59E0B")
plt.title("Loss over Epochs", fontsize=12, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Evaluation on Held-Out Test Set (10,000 Images)
Evaluation on the official 10,000 Fashion-MNIST test images with precision, recall, F1-score, and support.

In [ ]:
# Predictions on test set
prob_preds = model.predict(x_test, batch_size=128, verbose=1)
y_pred = np.argmax(prob_preds, axis=-1)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Held-out Test Loss:     {test_loss:.4f}")
print(f"Held-out Test Accuracy: {test_acc * 100:.2f}%")
print(f"Macro F1-Score:         {macro_f1 * 100:.2f}%")
print("\n" + classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

## 8. 10-Class Confusion Matrix Heatmap

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(f"Total items in confusion matrix: {cm.sum()} (must equal 10,000)")

fig, ax = plt.subplots(figsize=(9, 8), dpi=100)
cax = ax.imshow(cm, interpolation="nearest", cmap="Blues")
fig.colorbar(cax, fraction=0.046, pad=0.04)

tick_marks = np.arange(len(CLASS_NAMES))
ax.set_xticks(tick_marks)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(tick_marks)
ax.set_yticklabels(CLASS_NAMES)
ax.set_title("Fashion-MNIST 10-Class Test Confusion Matrix", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Predicted Label", fontweight="bold")
ax.set_ylabel("True Label", fontweight="bold")

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        val = cm[i, j]
        color = "white" if val > thresh else "black"
        ax.text(j, i, f"{val}", ha="center", va="center", color=color, fontsize=8)

plt.tight_layout()
plt.show()

## 9. Visualizing Real Misclassified Test Images

In [ ]:
misclassified_indices = np.where(y_pred != y_test)[0]
print(f"Total misclassified test images: {len(misclassified_indices)} / 10,000 ({len(misclassified_indices)/100:.2f}%)")

# Display first 15 misclassified items
plt.figure(figsize=(15, 8))
for i, idx in enumerate(misclassified_indices[:15]):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_test[idx].squeeze(), cmap="gray")
    true_label = CLASS_NAMES[y_test[idx]]
    pred_label = CLASS_NAMES[y_pred[idx]]
    conf = prob_preds[idx, y_pred[idx]] * 100
    plt.title(f"True: {true_label}\nPred: {pred_label} ({conf:.1f}%)", color="red", fontsize=9)
    plt.axis("off")
plt.suptitle("Sample Real Misclassified Test Images", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 10. Persisting Model & Exporting Artifacts

In [ ]:
output_dir = Path("../backend/models")
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "fashion_mnist_cnn.keras"
model.save(model_path)
print(f"Model saved successfully to {model_path.resolve()}")